# Reinforcement Learning Assignment 4

> This notebook follows the planning setting: the model $P$ and $R$ are known.

> Environment: **5x5 stochastic grid world**, discount **$\gamma = 0.95$**

> Actions: Up, Down, Left, Right with transition model **70% intended + 10% each other direction**

> Policy symbols: `^` (Up), `v` (Down), `<` (Left), `>` (Right)

> ASCII arrows are used to avoid Windows console encoding problems.

## What This Notebook Does
- Defines the MDP $\langle S, A, P, R, \gamma \rangle$
- Uses dynamic programming with full model knowledge
- Computes $v^\pi$ by iterative policy evaluation
- Computes $v^*$ and $\pi^*$ by value iteration
- Bonus: policy iteration (evaluate + improve) and compare with value iteration
- Explains why different reward settings change the optimal policy

In [10]:
import numpy as np
from collections import defaultdict

# Reproducibility for random initial policies in Policy Iteration
np.random.seed(42)

## 1) Environment Definition (MDP Setup)

> An MDP is defined by $\langle S, A, P, R, \gamma \rangle$.

> In this assignment we plan because the model $P$ and $R$ are known.

> The environment is fully observable, so the state is Markov.

### State Space $S$
- The world is a **5x5 grid**.
- Each state is a cell location `(row, col)`.
- Total states: $25$.

> Key states: `(0,0)` has reward `R1`, `(0,4)` has reward `R2`.

> Risky states: column 3 has `-1`, column 4 rows 1..4 have `-2`.

> These penalties influence the optimal policy under stochastic transitions.

### Reward Function $R$ (by column)
- Column 0: top cell is `R1`, remaining cells are `2`
- Column 1: all cells are `1`
- Column 2: all cells are `0`
- Column 3: all cells are `-1`
- Column 4: top cell is `R2`, remaining cells are `-2`

> This creates a tradeoff between **immediate safe reward** and **distant high reward**.

### Action Set $A$ and Transition Model $P$
- Actions: `Up, Down, Left, Right`
- Stochastic transitions: intended direction succeeds with `0.7`
- Each other direction occurs with `0.1`

> Example: if the agent selects `Right`, then:
- Right with prob `0.7`
- Left with prob `0.1`
- Down with prob `0.1`
- Up with prob `0.1`

> Wall rule: actions that hit the boundary keep the agent in place.

### Discount Factor $\gamma$
- We use $\gamma = 0.95$.
- The return is $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$.
- Larger $\gamma$ makes the agent more far-sighted; smaller $\gamma$ makes it more myopic.

In [11]:
class GridWorld:
    def __init__(self, R1, R2, gamma=0.95):
        self.grid_size = 5
        self.R1 = R1
        self.R2 = R2
        self.gamma = gamma

        # Action indices: 0=Up, 1=Down, 2=Left, 3=Right
        self.actions = ['Up', 'Down', 'Left', 'Right']
        self.action_deltas = {
            0: (-1, 0),
            1: (1, 0),
            2: (0, -1),
            3: (0, 1),
        }

        self.rewards = self._initialize_rewards()
        self.transitions = self._compute_transitions()

    def _initialize_rewards(self):
        rewards = np.zeros((self.grid_size, self.grid_size), dtype=float)

        # Column 0
        rewards[0, 0] = self.R1
        rewards[1:, 0] = 2

        # Column 1
        rewards[:, 1] = 1

        # Column 2
        rewards[:, 2] = 0

        # Column 3
        rewards[:, 3] = -1

        # Column 4
        rewards[0, 4] = self.R2
        rewards[1:, 4] = -2

        return rewards

    def _compute_transitions(self):
        transitions = {}

        # For each intended action, list (actual_action, probability).
        action_outcomes = {
            0: [(0, 0.7), (1, 0.1), (2, 0.1), (3, 0.1)],  # intended Up
            1: [(1, 0.7), (0, 0.1), (2, 0.1), (3, 0.1)],  # intended Down
            2: [(2, 0.7), (3, 0.1), (1, 0.1), (0, 0.1)],  # intended Left
            3: [(3, 0.7), (2, 0.1), (1, 0.1), (0, 0.1)],  # intended Right
        }

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                for action in range(4):
                    outcomes = []

                    for actual_action, prob in action_outcomes[action]:
                        next_row = row + self.action_deltas[actual_action][0]
                        next_col = col + self.action_deltas[actual_action][1]

                        # Wall handling: stay in place if out of bounds
                        if (next_row < 0 or next_row >= self.grid_size or
                            next_col < 0 or next_col >= self.grid_size):
                            next_row, next_col = row, col

                        outcomes.append(((next_row, next_col), prob))

                    transitions[(row, col, action)] = outcomes

        return transitions

    def get_reward(self, row, col):
        return self.rewards[row, col]

    def get_transitions(self, row, col, action):
        return self.transitions[(row, col, action)]

## 2) Helper Printing Functions

> These utilities only format output; they are not part of the RL theory.

> They mirror the lecture-style gridworld tables so you can interpret values and policies quickly.

### Helpers
- `print_reward_grid(...)`: shows the reward function $R(s)$
- `print_value_grid(...)`: shows state values $v(s)$ in table form
- `print_policy_grid(...)`: shows greedy actions using `^ v < >`

In [12]:
def print_reward_grid(rewards, R1, R2):
    print(f"\nGrid World Rewards (R1={R1}, R2={R2}):")
    print("+-------+-------+-------+-------+-------+")

    for row_idx, row in enumerate(rewards):
        row_str = "|"
        for col_idx, val in enumerate(row):
            if row_idx == 0 and col_idx == 0:
                row_str += f" R1={int(val):>2}   |"
            elif row_idx == 0 and col_idx == 4:
                row_str += f" R2={int(val):>2}   |"
            else:
                row_str += f" {int(val):>3}    |"
        print(row_str)

        if row_idx < 4:
            print("+-------+-------+-------+-------+-------+")

    print("+-------+-------+-------+-------+-------+")


def print_value_grid(values, title):
    print(f"\n{title}")
    print("-" * 50)
    for row in values:
        print("  ".join(f"{val:8.2f}" for val in row))


def print_policy_grid(policy, title):
    arrow_map = {0: '^', 1: 'v', 2: '<', 3: '>'}

    print(f"\n{title}")
    print("-" * 40)
    for row in policy:
        print("  ".join(arrow_map[a] for a in row))

## 3) Value Iteration

> Value iteration solves the **control** problem: find $v^*$ and $\pi^*$.

> It repeatedly applies the **Bellman optimality backup**.

> There is no explicit policy during updates; the policy is extracted after convergence.

### Bellman Optimality Equation
$$
v^*(s) = \max_{a \in A} \left[ R_s^a + \gamma \sum_{s' \in S} P_{ss'}^a v^*(s') \right]
$$

> Intuition: "best immediate reward + discounted best future".

### Value Iteration Update (Synchronous DP)
$$
v_{k+1}(s) \leftarrow \max_{a \in A} \left[ R_s^a + \gamma \sum_{s' \in S} P_{ss'}^a v_k(s') \right]
$$

> This is a **synchronous backup** over all states each iteration.

> Convergence is guaranteed by contraction mapping.

### Stopping Criterion
- We use `theta = 1e-6`.
- The max-norm error bound is
$$
\|v_k - v^*\|_\infty \le \frac{\theta}{1-\gamma} = \frac{10^{-6}}{0.05} = 2\times 10^{-5}
$$
- This is sufficient for stable policy extraction.

In [13]:
class ValueIteration:
    def __init__(self, grid_world, theta=1e-6):
        # theta=1e-6 ensures value accuracy within theta/(1-gamma)=2e-5,
        # which is sufficient for reliable policy extraction.
        self.grid = grid_world
        self.theta = theta
        self.values = np.zeros((self.grid.grid_size, self.grid.grid_size), dtype=float)
        self.policy = None
        self.iterations = 0

    def compute_action_value(self, row, col, action):
        q_val = 0.0
        reward = self.grid.get_reward(row, col)

        for (nr, nc), prob in self.grid.get_transitions(row, col, action):
            q_val += prob * (reward + self.grid.gamma * self.values[nr, nc])

        return q_val

    def iterate_once(self):
        new_values = np.zeros_like(self.values)
        max_delta = 0.0

        for r in range(self.grid.grid_size):
            for c in range(self.grid.grid_size):
                q_vals = [self.compute_action_value(r, c, a) for a in range(4)]
                new_values[r, c] = max(q_vals)
                max_delta = max(max_delta, abs(new_values[r, c] - self.values[r, c]))

        self.values = new_values
        self.iterations += 1
        return max_delta

    def solve(self, verbose=True):
        if verbose:
            print(f"\n{'=' * 60}")
            print(f"Value Iteration (R1={self.grid.R1}, R2={self.grid.R2})")
            print(f"{'=' * 60}")

        while True:
            delta = self.iterate_once()

            if verbose and ((self.iterations - 1) % 10 == 0 or delta < self.theta):
                print(f"Iteration {self.iterations}: Delta = {delta:.2e}")

            if delta < self.theta:
                if verbose:
                    print(f"Converged after {self.iterations} iterations\n")
                break

    def extract_policy(self):
        policy = np.zeros((self.grid.grid_size, self.grid.grid_size), dtype=int)

        for r in range(self.grid.grid_size):
            for c in range(self.grid.grid_size):
                q_vals = [self.compute_action_value(r, c, a) for a in range(4)]
                policy[r, c] = int(np.argmax(q_vals))

        self.policy = policy
        return policy

## 4) Bonus: Policy Iteration

> Policy iteration solves control by alternating **policy evaluation** and **policy improvement**.

> This directly matches the algorithm definition: evaluate $v^\pi$, then improve the policy greedily.

### Policy Evaluation (Bellman Expectation)
$$
v^\pi(s) = \sum_{a \in A} \pi(a|s) \left[ R_s^a + \gamma \sum_{s' \in S} P_{ss'}^a v^\pi(s') \right]
$$

> In code, we use **iterative policy evaluation** (synchronous backups).

> This converges to the unique fixed point of the Bellman expectation operator.

### Policy Improvement (Greedy)
$$
\pi_{new}(s) = \arg\max_{a \in A} \left[ R_s^a + \gamma \sum_{s' \in S} P_{ss'}^a v^\pi(s') \right]
$$

> Policy improvement guarantees $v^{\pi_{new}}(s) \ge v^\pi(s)$ for all $s$.

> Iterating evaluation and improvement converges to $\pi^*$.

> We print the initial random policy to confirm random initialization.

In [14]:
class PolicyIteration:
    def __init__(self, grid_world, initial_policy=None):
        self.grid = grid_world
        self.values = np.zeros((self.grid.grid_size, self.grid.grid_size), dtype=float)

        if initial_policy is None:
            self.policy = np.random.randint(0, 4, (self.grid.grid_size, self.grid.grid_size))
        else:
            self.policy = initial_policy.copy()

        self.iterations = 0

    def policy_evaluation(self, max_iter=200, eval_theta=1e-6):
        for _ in range(max_iter):
            new_values = np.zeros_like(self.values)
            max_delta = 0.0

            for r in range(self.grid.grid_size):
                for c in range(self.grid.grid_size):
                    a = int(self.policy[r, c])
                    reward = self.grid.get_reward(r, c)

                    v = 0.0
                    for (nr, nc), prob in self.grid.get_transitions(r, c, a):
                        v += prob * (reward + self.grid.gamma * self.values[nr, nc])

                    new_values[r, c] = v
                    max_delta = max(max_delta, abs(v - self.values[r, c]))

            self.values = new_values

            if max_delta < eval_theta:
                break

    def policy_improvement(self):
        improved = False
        new_policy = self.policy.copy()

        for r in range(self.grid.grid_size):
            for c in range(self.grid.grid_size):
                reward = self.grid.get_reward(r, c)
                action_values = []

                for a in range(4):
                    q = 0.0
                    for (nr, nc), prob in self.grid.get_transitions(r, c, a):
                        q += prob * (reward + self.grid.gamma * self.values[nr, nc])
                    action_values.append(q)

                best_a = int(np.argmax(action_values))
                if best_a != int(self.policy[r, c]):
                    improved = True
                    new_policy[r, c] = best_a

        self.policy = new_policy
        return improved

    def solve(self, verbose=True):
        arrow_map = {0: '^', 1: 'v', 2: '<', 3: '>'}

        if verbose:
            print(f"\n{'=' * 60}")
            print(f"Policy Iteration (R1={self.grid.R1}, R2={self.grid.R2})")
            print(f"{'=' * 60}")
            print("Starting from random policy...\n")

            print("Initial Random Policy:")
            print("-" * 40)
            for row in self.policy:
                print("  ".join(arrow_map[a] for a in row))
            print()

        while True:
            self.policy_evaluation()
            changed = self.policy_improvement()
            self.iterations += 1

            if verbose:
                status = "changed" if changed else "converged"
                print(f"Iteration {self.iterations}: Policy {status}")

            if not changed:
                if verbose:
                    print(f"Converged after {self.iterations} iterations\n")
                break

## 5) Explanation Helper

> This diagnostic section compares VI and PI outputs for each case and summarizes action patterns.

### Outputs Provided
- `Policies match: True/False`
- Frequency of greedy actions in the VI policy
- Edge behaviors (top row, bottom row, left column, right column)

> This is useful for explaining policy structure in your report.

In [15]:
def analyze_policy(vi_policy, pi_policy, R1, R2):
    arrow_map = {0: '^', 1: 'v', 2: '<', 3: '>'}

    print(f"\n{'=' * 60}")
    print(f"POLICY COMPARISON (R1={R1}, R2={R2})")
    print(f"{'=' * 60}")

    same = np.array_equal(vi_policy, pi_policy)
    print(f"Policies match: {same} [YES]" if same else f"Policies match: {same} [NO]")

    counts = defaultdict(int)
    for a in vi_policy.flatten():
        counts[arrow_map[int(a)]] += 1

    print("\nAction frequencies in VI optimal policy:")
    for k in sorted(counts):
        print(f"  {k}: {counts[k]} states")

    print("\nEdge behavior:")
    top = [arrow_map[int(a)] for a in vi_policy[0, :]]
    bottom = [arrow_map[int(a)] for a in vi_policy[-1, :]]
    left = [arrow_map[int(a)] for a in vi_policy[:, 0]]
    right = [arrow_map[int(a)] for a in vi_policy[:, -1]]

    print(f"  Top row: {' '.join(top)}")
    print(f"  Bottom row: {' '.join(bottom)}")
    print(f"  Left col: {' '.join(left)}")
    print(f"  Right col: {' '.join(right)}")

## 6) Run All Required Cases (and Bonus)

> Required cases:
1. `(R1, R2) = (100, 110)`
2. `(R1, R2) = (10, 100)`
3. `(R1, R2) = (1, 10)`
4. `(R1, R2) = (10, 15)`

> For each case, we run:
- Value Iteration (Bellman optimality backups)
- Policy Iteration (evaluate + improve)
- A VI vs PI comparison check

> This is the full DP planning loop applied to a known MDP model.

In [16]:
test_cases = [(100, 110), (10, 100), (1, 10), (10, 15)]

all_vi_policies = {}
all_pi_policies = {}
all_vi_iterations = {}
all_pi_iterations = {}

print('\n' + '=' * 70)
print('PART 1: VALUE ITERATION')
print('=' * 70)

for R1, R2 in test_cases:
    grid = GridWorld(R1, R2)
    print_reward_grid(grid.rewards, R1, R2)

    vi = ValueIteration(grid)
    vi.solve(verbose=True)
    vi_policy = vi.extract_policy()

    all_vi_policies[(R1, R2)] = vi_policy
    all_vi_iterations[(R1, R2)] = vi.iterations

    print_value_grid(vi.values, f'Value Function (R1={R1}, R2={R2})')
    print_policy_grid(vi_policy, f'Optimal Policy - Value Iteration (R1={R1}, R2={R2})')

print('\n' + '=' * 70)
print('PART 2: BONUS - POLICY ITERATION')
print('=' * 70)

for R1, R2 in test_cases:
    grid = GridWorld(R1, R2)
    pi = PolicyIteration(grid)
    pi.solve(verbose=True)

    all_pi_policies[(R1, R2)] = pi.policy
    all_pi_iterations[(R1, R2)] = pi.iterations

    print_value_grid(pi.values, f'Value Function (R1={R1}, R2={R2})')
    print_policy_grid(pi.policy, f'Optimal Policy - Policy Iteration (R1={R1}, R2={R2})')

print('\n' + '=' * 70)
print('PART 3: VI vs PI CONVERGENCE CHECK')
print('=' * 70)

for R1, R2 in test_cases:
    analyze_policy(all_vi_policies[(R1, R2)], all_pi_policies[(R1, R2)], R1, R2)


PART 1: VALUE ITERATION

Grid World Rewards (R1=100, R2=110):
+-------+-------+-------+-------+-------+
| R1=100   |   1    |   0    |  -1    | R2=110   |
+-------+-------+-------+-------+-------+
|   2    |   1    |   0    |  -1    |  -2    |
+-------+-------+-------+-------+-------+
|   2    |   1    |   0    |  -1    |  -2    |
+-------+-------+-------+-------+-------+
|   2    |   1    |   0    |  -1    |  -2    |
+-------+-------+-------+-------+-------+
|   2    |   1    |   0    |  -1    |  -2    |
+-------+-------+-------+-------+-------+

Value Iteration (R1=100, R2=110)
Iteration 1: Delta = 1.10e+02
Iteration 11: Delta = 4.76e+01
Iteration 21: Delta = 2.84e+01
Iteration 31: Delta = 1.70e+01
Iteration 41: Delta = 1.02e+01
Iteration 51: Delta = 6.08e+00
Iteration 61: Delta = 3.64e+00
Iteration 71: Delta = 2.18e+00
Iteration 81: Delta = 1.30e+00
Iteration 91: Delta = 7.79e-01
Iteration 101: Delta = 4.66e-01
Iteration 111: Delta = 2.79e-01
Iteration 121: Delta = 1.67e-01
Iterati

## 7) Case-by-Case Interpretation

> Use these explanations to justify *why* the greedy actions appear in each cell.

### What to Discuss in Your Report
- How the return $G_t$ trades off immediate vs delayed reward
- How discounting ($\gamma$) makes distant rewards less attractive
- How stochastic transitions increase risk of entering negative columns
- Why greedy actions follow the Bellman optimality equation locally
- How penalties in columns 3 and 4 shift policies upward or leftward

In [17]:
insights = {
    (100, 110): '''
CASE (R1=100, R2=110):
Both corner rewards are huge and close in value.

Notable cell: (0,1) often points LEFT.
Why? The left corner reward 100 is only one step away, so the immediate discounted pull is strong
(roughly 0.95*100 = 95 in one-step lookahead logic), while moving right pushes the agent toward
columns with -1 and possibly -2 outcomes under slip risk.

Because transitions are stochastic (70/10/10/10), intended right moves can slip down into worse rows
or fail to progress as planned. So even though R2 is bigger by 10, the safer and nearer reward can
locally dominate some top-left states.

Gamma=0.95 means distance still matters: each extra step multiplies value by 0.95. A reward that is
a few cells farther can lose much of its advantage, especially when path risk includes negative columns.

Negative columns shape policy boundaries:
- column 3 (-1) acts like a soft barrier
- column 4 rows 1-4 (-2) create stronger avoidance pressure
This is why right-edge states usually point upward quickly to escape -2 rows.
''',

    (10, 100): '''
CASE (R1=10, R2=100):
Now the top-right reward is overwhelmingly larger. Most states should flow right/up.

Even with discounting, the gap is massive: a farther reward of 100 still dominates a nearer reward of 10.
So the policy aggressively heads to the top-right despite stochastic slips.

Stochastic risk is still present near -1 and -2 columns, but here the expected long-term gain from R2
is large enough to absorb occasional penalty exposure.

Gamma=0.95 does reduce urgency with distance, but not enough to overturn a 90-point reward gap.
As a result, leftward actions become rare except where local tie-breaking or boundary effects appear.
''',

    (1, 10): '''
CASE (R1=1, R2=10):
R2 is still better, but absolute rewards are small, so penalties and slip risk matter more proportionally.

In lower-right areas, moving right can expose the agent to -2 states under stochastic outcomes,
so some cells prefer moving up first (to reduce expected penalty time) before final approach to R2.

Compared with (10,100), the rightward bias is weaker. You often see a conservative pattern:
go up to safer rows, then go right near the top where penalty risk is smaller.

Gamma=0.95 means a few extra steps significantly reduce benefit when rewards are small.
So tiny local costs/risks can change action choice more easily in this case.
''',

    (10, 15): '''
CASE (R1=10, R2=15):
This is the closest contest. The 5-point difference is modest, so distance and risk decide many states.

Some cells that look like they should go right may choose up (or even left in edge cases) because:
- one target is closer under gamma discount
- stochastic slips can send the agent into -1/-2 columns
- expected Q-values become very close, making boundaries sensitive

With gamma=0.95, each extra step is costly enough that nearby safe returns can rival slightly larger but
farther rewards. This is why policy can look more balanced and less directional than in extreme cases.

Negative columns play a decisive role here: because reward gap is small, avoiding expected penalty exposure
often matters as much as chasing the larger corner reward.
''',
}

print('\n' + '=' * 70)
print('CASE-BY-CASE EXPLANATIONS')
print('=' * 70)

for case in test_cases:
    R1, R2 = case
    print(f'\nR1={R1}, R2={R2}')
    print(insights[case])


CASE-BY-CASE EXPLANATIONS

R1=100, R2=110

CASE (R1=100, R2=110):
Both corner rewards are huge and close in value.

Notable cell: (0,1) often points LEFT.
Why? The left corner reward 100 is only one step away, so the immediate discounted pull is strong
(roughly 0.95*100 = 95 in one-step lookahead logic), while moving right pushes the agent toward
columns with -1 and possibly -2 outcomes under slip risk.

Because transitions are stochastic (70/10/10/10), intended right moves can slip down into worse rows
or fail to progress as planned. So even though R2 is bigger by 10, the safer and nearer reward can
locally dominate some top-left states.

Gamma=0.95 means distance still matters: each extra step multiplies value by 0.95. A reward that is
a few cells farther can lose much of its advantage, especially when path risk includes negative columns.

Negative columns shape policy boundaries:
- column 3 (-1) acts like a soft barrier
- column 4 rows 1-4 (-2) create stronger avoidance pressure
Th

## 8) Final Summary Table

> The table is filled from actual run counts and summarizes convergence behavior.

### How to Interpret
- VI uses repeated Bellman optimality backups on $v_k$
- PI alternates policy evaluation and improvement
- PI typically needs fewer *outer* iterations, but each evaluation is heavier

In [18]:
def dominant_direction_label(policy):
    # Count action frequencies to summarize policy direction
    counts = defaultdict(int)
    for a in policy.flatten():
        counts[int(a)] += 1

    up = counts[0]
    right = counts[3]

    if up >= 18 and right <= 5:
        return 'Up'
    if up >= 12 and right >= 6:
        return 'Right+Up'
    return 'Mixed'

print('\n' + '=' * 70)
print('CONVERGENCE & SUMMARY TABLE')
print('=' * 70)

print('+==============+========+==================+===============+')
print('| Case         | VI     | PI Iterations    | Dominant Dir  |')
print('|              | Iters  | (from random)    |               |')
print('+==============+========+==================+===============+')

for R1, R2 in test_cases:
    vi_iters = all_vi_iterations[(R1, R2)]
    pi_iters = all_pi_iterations[(R1, R2)]
    dom = dominant_direction_label(all_vi_policies[(R1, R2)])
    case_text = f'R1={R1:>3},R2={R2:<3}'
    print(f'| {case_text:<12} | {vi_iters:>6} | {pi_iters:>16} | {dom:<13} |')

print('+==============+========+==================+===============+')

print('\nNotes:')
print('- Policy Iteration should converge to the same optimal policy as Value Iteration.')
print('- PI usually needs far fewer outer iterations than VI in this grid.')


CONVERGENCE & SUMMARY TABLE
+==============+========+==================+===============+
| Case         | VI     | PI Iterations    | Dominant Dir  |
|              | Iters  | (from random)    |               |
+==============+========+==================+===============+
| R1=100,R2=110 |    356 |                5 | Up            |
| R1= 10,R2=100 |    354 |                4 | Right+Up      |
| R1=  1,R2=10  |    308 |                5 | Up            |
| R1= 10,R2=15  |    317 |                4 | Up            |
+==============+========+==================+===============+

Notes:
- Policy Iteration should converge to the same optimal policy as Value Iteration.
- PI usually needs far fewer outer iterations than VI in this grid.


## 9) What To Submit (Checklist)

> Use this checklist for your final submission.

1. For each case, show the reward grid, value grid, and policy grid.
2. Explain policy behavior using Bellman optimality and discounting.
3. Mention stochastic transition risk near the negative reward columns.
4. Bonus: show policy iteration starts random and converges to the same $\pi^*$ as VI.
5. Include the final summary table with actual iteration counts.